In [22]:
import numpy as np
import tensorflow as tf

from tensorflow.keras import layers, models
from tensorflow import keras

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.16.2


In [23]:
texts = [
    "Haseeb say's: life is all about ups and downs.",
    "Uzair says: Its a very hot day.",
    "There are five type of topologies: Bus topology, star topology, Ring topology, Mesh topology, Tree topology."
]

texts

["Haseeb say's: life is all about ups and downs.",
 'Uzair says: Its a very hot day.',
 'There are five type of topologies: Bus topology, star topology, Ring topology, Mesh topology, Tree topology.']

In [24]:
vectorizer = layers.TextVectorization(
    output_mode="int",
    standardize="lower_and_strip_punctuation",
)

vectorizer.adapt(texts)

vocabulary = vectorizer.get_vocabulary()

print("Vocabulary:")
for index, word in enumerate(vocabulary):
    print(index, "->", word)

Vocabulary:
0 -> 
1 -> [UNK]
2 -> topology
3 -> says
4 -> very
5 -> uzair
6 -> ups
7 -> type
8 -> tree
9 -> topologies
10 -> there
11 -> star
12 -> ring
13 -> of
14 -> mesh
15 -> life
16 -> its
17 -> is
18 -> hot
19 -> haseeb
20 -> five
21 -> downs
22 -> day
23 -> bus
24 -> are
25 -> and
26 -> all
27 -> about
28 -> a


In [31]:
word = "topology"

token_id = vectorizer([word])

print("Word:", word)
print("Token ID:", token_id.numpy())

Word: topology
Token ID: [[2]]


In [32]:
vocab_size = len(vocabulary)
embedding_dim = 8

embedding = layers.Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim
)

In [33]:
vector = embedding(token_id)

print("Word:", word)
print("Token ID:", token_id.numpy())
print("Vector:")
print(vector.numpy())

Word: topology
Token ID: [[2]]
Vector:
[[[-0.01842117  0.00797734 -0.02877216  0.008146   -0.03209046
   -0.0197826   0.03818989 -0.00439124]]]


In [34]:
word_ids = vectorizer(texts)

word_vectors = embedding(word_ids)

print(word_ids.numpy())
print(word_vectors.shape)

[[19  3 15 17 26 27  6 25 21  0  0  0  0  0  0  0]
 [ 5  3 16 28  4 18 22  0  0  0  0  0  0  0  0  0]
 [10 24 20  7 13  9 23  2 11  2 12  2 14  2  8  2]]
(3, 16, 8)


In [35]:
word_ids = vectorizer(texts)

word_vectors = embedding(word_ids)

for word, token_id, vector in zip(
    texts,
    word_ids.numpy().flatten(),
    word_vectors.numpy().reshape(len(texts), -1)
):
    print(f"{word:10} ID={token_id}  VECTOR={vector}")

Haseeb say's: life is all about ups and downs. ID=19  VECTOR=[-3.1795740e-02 -4.5687269e-02 -1.2976456e-02  7.0341341e-03
  4.6922453e-03  3.5854962e-02 -1.8466759e-02  4.4260476e-02
 -1.6350724e-02 -4.3305684e-02 -1.6829062e-02 -2.4541402e-02
 -5.4771677e-03  8.9741349e-03  4.8881780e-02 -3.1399347e-02
 -3.4637965e-02 -4.8766315e-02 -3.2270327e-02 -4.9300443e-02
 -1.4024507e-02 -6.7838430e-03  3.9059307e-02  3.3470009e-02
  1.1504043e-02  3.0271400e-02  3.4294102e-02  1.4764976e-02
  3.4066070e-02  2.6122443e-03  3.5754416e-02  2.1847252e-02
 -1.0954071e-02 -3.5107434e-02  3.0595545e-02  4.9109254e-02
  3.8381759e-02 -3.6869742e-02  1.5381526e-02 -1.2809921e-02
  4.2101610e-02 -1.4084898e-02 -2.6747381e-02  3.2460224e-02
 -6.2609911e-03 -1.4667999e-02 -1.8517505e-02  2.6723336e-02
 -1.7307624e-03 -1.6605355e-02 -4.7869205e-02 -4.5854259e-02
 -4.4544518e-02 -3.7600566e-02 -4.3682363e-02  4.0747132e-02
  4.2947683e-02 -1.7019361e-04  2.6038695e-02  1.8608458e-03
 -2.0099962e-02  1.47644

In [38]:
def find_closest_word(query_vector):
    all_vectors = embedding.weights[0].numpy()

    distances = np.linalg.norm(
        all_vectors - query_vector,
        axis=1
    )

    closest_id = np.argmin(distances)

    return vocabulary[closest_id], closest_id

def find_closest_words(query_vector, top_n=5):
    all_vectors = embedding.weights[0].numpy()

    distances = np.linalg.norm(
        all_vectors - query_vector,
        axis=1
    )

    closest_ids = np.argsort(distances)[:top_n]

    return [(vocabulary[i], i, distances[i]) for i in closest_ids]

In [37]:
original_vector = embedding(
    vectorizer(["topology"])
)[0]

word_found, token_id_found = find_closest_word(
    original_vector.numpy()
)

print("Input vector:")
print(original_vector.numpy())

print("\nClosest word:")
print(word_found)

print("Token ID:")
print(token_id_found)

Input vector:
[[-0.01842117  0.00797734 -0.02877216  0.008146   -0.03209046 -0.0197826
   0.03818989 -0.00439124]]

Closest word:
topology
Token ID:
2


In [39]:
original_vector = embedding(
    vectorizer(["topology"])
)[0]

results = find_closest_words(
    original_vector.numpy(), top_n=5
)

print("Top matches:")
for word, token_id, dist in results:
    print(f"{word:12} ID={token_id:<4} distance={dist:.4f}")

Top matches:
topology     ID=2    distance=0.0000
are          ID=24   distance=0.0568
uzair        ID=5    distance=0.0671
             ID=0    distance=0.0690
its          ID=16   distance=0.0737


In [40]:
# 1. Build skip-gram pairs (target word -> nearby word) from your sentences
window_size = 2
sequences = vectorizer(texts).numpy()

pairs = []
for seq in sequences:
    seq = [t for t in seq if t != 0]  # drop padding
    for i, target in enumerate(seq):
        start = max(0, i - window_size)
        end = min(len(seq), i + window_size + 1)
        for j in range(start, end):
            if j != i:
                pairs.append((target, seq[j]))

targets = np.array([p[0] for p in pairs])
contexts = np.array([p[1] for p in pairs])

In [41]:
# 2. Train the SAME embedding layer to predict context from target
skipgram_model = models.Sequential([
    embedding,
    layers.GlobalAveragePooling1D(),
    layers.Dense(vocab_size, activation="softmax")
])

skipgram_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
skipgram_model.fit(targets.reshape(-1, 1), contexts, epochs=200, verbose=0)

In [44]:
# 3. Re-check closest words — embedding object is the same one, now trained
original_vector = embedding(vectorizer(["topology"]))[0]
results = find_closest_words(original_vector.numpy(), top_n=5)
for word, token_id, dist in results:
    print(f"{word:12} ID={token_id:<4} distance={dist:.4f}")

topology     ID=2    distance=0.0000
ring         ID=12   distance=0.4829
mesh         ID=14   distance=0.5319
tree         ID=8    distance=0.8909
star         ID=11   distance=0.9490
